# Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# auto-reload pygenelab when its .py files change (no need to restart the kernel)
%load_ext autoreload
%autoreload 2

import pygenelab as pgl
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from statsmodels.stats.multitest import multipletests

# Load Dataset

In [ ]:
# mice dataset path
UPDATED_MICE_DATASET_PATH = "/ix/djishnu/Akanksha/datasets/ERCC1_KO/objects/objects/adata_harmony_v2_re_annot.h5ad"

# load adata
adata = sc.read_h5ad(UPDATED_MICE_DATASET_PATH)

In [ ]:
# columns to use
celltype_col = "re_annot"
condition_col = "condition"

# groups to compare
group1 = "WT"
group2 = "KO"

layer = None

In [ ]:
cell_types = adata.obs["re_annot"].unique().to_list()
cell_types

In [ ]:
SEX = None
if SEX == "F" or SEX == "M":
    adata = pgl.subset_adata_by_obs(adata, filters={"sex": SEX})
adata.obs["sex"].value_counts()

# Calculate Transcriptional Noise

In [ ]:
all_noise = []
all_summary = []

for celltype in cell_types:
    print(f"processing: {celltype}")

    # subset one cell type
    adata_ct = adata[
        adata.obs[celltype_col] == celltype
    ].copy()

    # calculate noise
    noise_df, summary = pgl.calculate_noise_one_celltype(
        adata_ct=adata_ct,
        condition_col=condition_col,
        group1=group1,
        group2=group2,
        layer=layer,
        max_cells=300,
        min_cells=10,
        n_bins=10,
        bottom_frac=0.10,
        random_state=1
    )

    # store summary
    summary["celltype"] = celltype
    all_summary.append(summary)

    # store per-cell noise values
    if noise_df is not None:
        noise_df["celltype"] = celltype
        all_noise.append(noise_df)

# combine results
noise_all_df = pd.concat(all_noise, ignore_index=True)
summary_df = pd.DataFrame(all_summary)

# adjust p-values
done_mask = summary_df["status"] == "done"

summary_df.loc[done_mask, "padj"] = multipletests(
    summary_df.loc[done_mask, "pval"],
    method="fdr_bh"
)[1]

summary_df["significant"] = summary_df["padj"] < 0.05

# sort summary
summary_df = summary_df.sort_values(
    f"log2_{group2}_over_{group1}",
    ascending=False,
    na_position="last"
)

summary_df

# Plot

In [ ]:
fig, ax = pgl.plot_transcriptional_noise(
    noise_all_df=noise_all_df,
    celltype_col="celltype",
    condition_col="condition",
    noise_col="noise",
    celltype_order=cell_types,
    group_order=(group1, group2),
    figsize=(14, 5),
    title=f"{SEX} WT vs KO Transcriptional Noise by Cell Type"
)

plt.show()

# Save Plot

In [ ]:
output_dir = Path("Output/Transcriptional_Heterogeneity")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# save plot
png_path = output_dir / f"{SEX}_WT_vs_KO_Transcriptional_Noise_PNG.png"
svg_path = output_dir / f"{SEX}_WT_vs_KO_Transcriptional_Noise_SVG.svg"

pgl.save_fig_as_png(fig, png_path)
pgl.save_editable_svg(fig, svg_path)

In [ ]:
# save stats
stats_df = summary_df[["celltype", "n_WT", "n_KO", "n_low_cv_genes", "mean_noise_WT", "mean_noise_KO", "log2_KO_over_WT"]]

stats_png_path = output_dir / f"{SEX}_WT_vs_KO_Transcriptional_Noise_Stats_PNG.png"
stats_svg_path = output_dir / f"{SEX}_WT_vs_KO_Transcriptional_Noise_Stats_SVG.svg"

stats_fig = pgl.save_df_table_image(stats_df)

pgl.save_fig_as_png(stats_fig, stats_png_path)
pgl.save_editable_svg(stats_fig, stats_svg_path)